# 模块 A · 读入 10X Xenium（单细胞级，靶向 panel）   [Python]

- 对应课程：第19课

**对应的流程脚本**：`scripts/A_load/load_xenium.py`（代码相同，流程运行的是 .py）

**用法**：先把下面第一个代码格里的 `project_dir` 改成项目目录，然后从上到下逐格运行。

In [ ]:
# 步骤：切换到项目目录（里面有 config.yaml、samples.tsv 和 results/）
# 输入：project_dir（改成自己的项目目录）
# 输出：无（之后的代码都以项目目录为当前目录）
import os
project_dir = "."      # 例如 "/data/my_visium_project"
os.chdir(project_dir)
os.getcwd()

## 1. 导入包

In [ ]:
# 步骤：导入包
# 输入：无
# 输出：无
from pathlib import Path
import pandas as pd
import scanpy as sc
import yaml

## 2. 读取配置和样本表

In [ ]:
# 步骤：读取配置和样本表
# 输入：config.yaml；samples.tsv（path 指向 Xenium 输出目录）
# 输出：cfg、samples、out_dir
cfg = yaml.safe_load(open("config.yaml"))
out_dir = Path(cfg["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)
samples = pd.read_csv(cfg["samples"], sep="\t")
samples

这一步得到：cfg、samples、out_dir

## 3. 逐个样本读入细胞 x 基因矩阵和细胞中心坐标（Xenium 坐标本身就是 µm）

In [ ]:
# 步骤：逐个样本读入细胞 x 基因矩阵和细胞中心坐标（Xenium 坐标本身就是 µm）
# 输入：cell_feature_matrix.h5、cells.parquet
# 输出：adatas（样本名 -> AnnData），obsm['spatial'] = obsm['spatial_um']
adatas = {}
for row in samples.itertuples():
    adata = sc.read_10x_h5(Path(row.path) / "cell_feature_matrix.h5")    # 只保留 Gene Expression，去掉对照探针
    cells = pd.read_parquet(Path(row.path) / "cells.parquet").set_index("cell_id")
    adata.obs = cells.loc[adata.obs_names]
    adata.obs["condition"] = row.condition
    adata.obsm["spatial_um"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy()
    adata.obsm["spatial"] = adata.obsm["spatial_um"]
    adatas[row.sample] = adata

这一步得到：adatas（样本名 -> AnnData），obsm['spatial'] = obsm['spatial_um']

## 4. 合并样本，保存原始 counts

In [ ]:
# 步骤：合并样本，保存原始 counts
# 输入：adatas
# 输出：adata（细胞 x 基因）；layers['counts']
adata = sc.concat(adatas, label="sample", index_unique="_")
adata.layers["counts"] = adata.X.copy()
adata

这一步得到：adata（细胞 x 基因）；layers['counts']

## 5. 保存

In [ ]:
# 步骤：保存
# 输入：adata
# 输出：results/01_raw.h5ad
adata.write_h5ad(out_dir / "01_raw.h5ad")

这一步得到：results/01_raw.h5ad